In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

print("All libraries imported successfully!")

All libraries imported successfully!


In [19]:
df = pd.read_csv("student_performance.csv")
print("Dataset loaded. Total rows and columns:", df.shape)

Dataset loaded. Total rows and columns: (100, 8)


In [20]:
df.head()

,Student_ID,Gender,Study_Hours,Attendance_Percent,Sleep_Hours,Previous_Marks,Internet_Access,Final_Marks
0,S001,Female,7.2,71.0,8.0,56.0,Yes,68.0
1,S002,Female,4.5,NaN,8.9,84.0,Yes,67.0
2,S003,Female,7.9,77.0,6.8,35.0,Yes,55.0
3,S004,Male,6.6,86.0,5.1,88.0,Yes,80.0
4,S005,Female,NaN,75.0,4.6,67.0,Yes,45.0


In [21]:
# Keep the 4 features and Final_Marks (needed to make the target)
df = df[["Study_Hours", "Attendance_Percent", "Previous_Marks", "Sleep_Hours", "Final_Marks"]].copy()
df = df.rename(columns={"Study_Hours": "Hours_Studied"})

# Use the SAME pass mark as Task 3
pass_mark = 40
df["Final_Result"] = df["Final_Marks"].apply(lambda m: "Pass" if m >= pass_mark else "Fail")
df = df.drop("Final_Marks", axis=1)

X = df.drop("Final_Result", axis=1)   # features
y = df["Final_Result"]                # target

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print()
print(y.value_counts())

Shape of X: (100, 4)
Shape of y: (100,)

Final_Result
Pass    95
Fail     5
Name: count, dtype: int64


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (80, 4)
X_test : (20, 4)
y_train: (80,)
y_test : (20,)


In [23]:
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Preprocessing done!")
print("Training data shape:", X_train_scaled.shape)
print("Testing data shape :", X_test_scaled.shape)

Preprocessing done!
Training data shape: (80, 4)
Testing data shape : (20, 4)


In [24]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

Model trained successfully!


In [25]:
y_pred = model.predict(X_test_scaled)

print("Predictions for the testing data:")
print(y_pred)

Predictions for the testing data:
['Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass'
 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass']


In [26]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
results["Correct?"] = results["Actual"] == results["Predicted"]

results

,Actual,Predicted,Correct?
0,Pass,Pass,True
1,Pass,Pass,True
2,Pass,Pass,True
3,Pass,Pass,True
4,Pass,Pass,True
5,Pass,Pass,True
6,Pass,Pass,True
7,Pass,Pass,True
8,Pass,Pass,True
9,Pass,Pass,True


In [27]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")
print()

print("Confusion Matrix (rows = actual, columns = predicted):")
cm = confusion_matrix(y_test, y_pred, labels=["Fail", "Pass"])
print(pd.DataFrame(cm, index=["Actual Fail", "Actual Pass"], columns=["Predicted Fail", "Predicted Pass"]))

Accuracy: 95.0 %

Confusion Matrix (rows = actual, columns = predicted):
             Predicted Fail  Predicted Pass
Actual Fail               0               1
Actual Pass               0              19


In [28]:
weights = pd.Series(model.coef_[0], index=X.columns).round(3)
print("Weight of each feature (positive = pushes towards Pass):")
print(weights)

Weight of each feature (positive = pushes towards Pass):
Hours_Studied         1.030
Attendance_Percent    0.552
Previous_Marks        0.786
Sleep_Hours          -0.476
dtype: float64


In [29]:
# Hours_Studied, Attendance_Percent, Previous_Marks, Sleep_Hours
new_student = pd.DataFrame([[6.0, 80.0, 65.0, 7.0]], columns=X.columns)

new_scaled = scaler.transform(imputer.transform(new_student))
prediction = model.predict(new_scaled)
probability = model.predict_proba(new_scaled)

print("Prediction for new student:", prediction[0])
print("Probability [Fail, Pass]  :", probability.round(3))

Prediction for new student: Pass
Probability [Fail, Pass]  : [[0.01 0.99]]
